In [2]:
# ViT vs CNN

# Loading ViT model
from transformers import ViTForImageClassification, ViTImageProcessor

model_name = "google/vit-base-patch16-224-in21k"

processor = ViTImageProcessor.from_pretrained(model_name)
model = ViTForImageClassification.from_pretrained(
    model_name,
    num_labels=10
)

Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
# Loading CNN model
from torchvision import models
import torch

restnet = models.resnet18(weights=True)
restnet.fc = torch.nn.Linear(restnet.fc.in_features, 10)

c:\Personal Projects\ViT_vs_CNN\venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [20]:
# Loading Dataset
from datasets import load_dataset

dataset = load_dataset("cifar10")
print(dataset.shape)

# Lets start with only 1000 samples :D
train_data = dataset["train"].shuffle(seed=42).select(range(1000))
test_data = dataset["test"]

print(train_data)
print(test_data)

{'train': (50000, 2), 'test': (10000, 2)}
Dataset({
    features: ['img', 'label'],
    num_rows: 1000
})
Dataset({
    features: ['img', 'label'],
    num_rows: 10000
})


In [14]:
# We need to preprocess the images for the ViT model
def preprocess_images(examples):
    inputs = processor(
        examples['img'],
        return_tensors="pt"
    )

    inputs['labels'] = examples['label']

    return inputs

train_data = train_data.with_transform(preprocess_images)
test_data = test_data.with_transform(preprocess_images)

In [15]:
# Fine-tuning ViT model
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./vit-small",
    learning_rate=2e-4,
    per_device_train_batch_size=32,
    num_train_epochs=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
)

trainer.train()

KeyError: 'img'